# CNN Cross-Entropy Testing for DNA Motifs

This notebook tests the cross-entropy performance of CNN models on DNA motifs, providing the same functionality as the BERT cross-entropy test but adapted for convolutional neural networks.

In [ ]:
import sys
sys.path.append("..")

In [ ]:
# Load CNN model and test proper MLM inference: mask motifs and predict them
import torch, pandas as pd, re, torch.nn.functional as F, os
from modules.masked_cnn import MaskedCNNConfig, DNACNNForMaskedLM
from modules.dna_tokenizer import DNATokenizer
from safetensors.torch import load_file

os.chdir('/home/shishir-sunar/InsDelGLM')

# Load CNN model
with open("model/CNN_baseline_100k_1e-4/vocab.txt", "r") as f:
    vocab = [line.strip() for line in f]
tokenizer = DNATokenizer(vocab=vocab, deletion_token=False)

# Load CNN config and model
config = MaskedCNNConfig.from_json_file("model/CNN_baseline_100k_1e-4/config_cnn.json")
model = DNACNNForMaskedLM(config)
model.load_state_dict(load_file("model/CNN_baseline_100k_1e-4/model.safetensors"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()

test_data = pd.read_parquet("data/simulated/new_baseline_test.parquet")

# Read motifs from file
with open("data/simulated/new_motifs.txt", "r") as f:
    motifs = [line.strip() for line in f]

mask_id = tokenizer._convert_token_to_id('[MASK]')
cls_id = tokenizer._convert_token_to_id('[CLS]')
sep_id = tokenizer._convert_token_to_id('[SEP]')

print(f"Loaded CNN model with {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Model config: {config}")
print(f"Test data shape: {test_data.shape}")
print(f"Motifs: {motifs}")

def proper_mlm_test(seq, model, motifs, tokenizer, device):
    """Proper MLM for CNN: mask motifs and predict them (like training)"""
    losses = []
    
    for motif in motifs:
        for match in re.finditer(motif, seq):
            start, end = match.span()
            
            # Tokenize sequence properly with [CLS] and [SEP]
            tokens = [tokenizer._convert_token_to_id(c) for c in seq if tokenizer._convert_token_to_id(c) is not None]
            input_ids = [cls_id] + tokens + [sep_id]
            
            # Mask the motif positions (offset by 1 for [CLS])
            masked_input = input_ids.copy()
            motif_start_idx = start + 1  # +1 for [CLS] token
            motif_tokens = [tokenizer._convert_token_to_id(c) for c in motif]
            
            # Replace motif tokens with [MASK]
            for i in range(len(motif)):
                if motif_start_idx + i < len(masked_input) - 1:  # Don't mask [SEP]
                    masked_input[motif_start_idx + i] = mask_id
            
            # Get predictions from CNN
            with torch.no_grad():
                logits = model(torch.tensor([masked_input], device=device), return_dict=True).logits[0]
            
            # Calculate loss only on masked positions
            pred_logits = logits[motif_start_idx:motif_start_idx + len(motif_tokens)]
            ground_truth = torch.tensor(motif_tokens, device=device)
            
            if len(pred_logits) == len(ground_truth):
                loss = F.cross_entropy(pred_logits, ground_truth)
                losses.append(loss.item())
                print(f"Motif '{motif}' at {start}-{end}: loss = {loss:.4f}")
    
    return sum(losses) / len(losses) if losses else 0

# Test
seq = test_data.iloc[5]['sequences']
print(f"\nSequence: {seq}")
print(f"Motifs: {motifs}")
print("-" * 40)
loss = proper_mlm_test(seq, model, motifs, tokenizer, device)
print(f"CNN Proper MLM cross-entropy at motifs: {loss:.4f}")

In [ ]:
# Comprehensive cross-entropy analysis for CNN model
import matplotlib.pyplot as plt
import numpy as np

def test_motifs_combined_cnn(seq, model, tokenizer, motif_list, device):
    """Test multiple motifs by masking ONE token at a time and averaging for CNN"""
    tokens = [tokenizer._convert_token_to_id(c) for c in seq if tokenizer._convert_token_to_id(c) is not None]
    cls_id = tokenizer._convert_token_to_id('[CLS]')
    sep_id = tokenizer._convert_token_to_id('[SEP]')
    mask_id = tokenizer._convert_token_to_id('[MASK]')
    input_ids = [cls_id] + tokens + [sep_id]
    
    # Find all motif positions
    motif_positions = []  # List of (start, end, motif_tokens)
    for motif in motif_list:
        for m in re.finditer(motif, seq):
            start, end = m.span()
            motif_tokens = [tokenizer._convert_token_to_id(c) for c in motif]
            motif_positions.append((start, end, motif_tokens))
    
    if not motif_positions:
        return None
    
    # Collect losses for each individual masked position
    all_losses = []
    
    for start, end, motif_tokens in motif_positions:
        for i, token in enumerate(motif_tokens):
            pos = start + 1 + i  # +1 for [CLS] token
            if pos >= len(input_ids) - 1:  # Skip if position is invalid
                continue
            
            # Create masked input with ONLY this position masked
            masked = input_ids.copy()
            masked[pos] = mask_id
            
            # Get predictions from CNN
            with torch.no_grad():
                output = model(torch.tensor([masked], device=device), return_dict=True)
                logits = output.logits[0]
            
            # Calculate loss for this single position
            pred_logit = logits[pos].unsqueeze(0)  # Shape: (1, vocab_size)
            truth_token = torch.tensor([token], device=device)  # Shape: (1,)
            
            loss = F.cross_entropy(pred_logit, truth_token)
            all_losses.append(loss.item())
    
    # Return average loss across all individual masked positions
    return sum(all_losses) / len(all_losses) if all_losses else None

def analyze_dataset_cnn(data, model, tokenizer, device):
    """Analyze all sequences with CNN model - categorize by motif presence"""
    results = {'motif_a': [], 'motif_b': [], 'both': [], 'background': []}
    avg_motif_len = int((len(motifs[0]) + len(motifs[1])) / 2)
    
    for count, (idx, row) in enumerate(data.iterrows()):
        if count % 10 == 0:
            print(f"Processing {count}...")
        
        if count > 100:  # Limit to first 100 sequences for speed
            break
        seq = row['sequences']
        
        # Check which motifs are present
        has_a = motifs[0] in seq
        has_b = motifs[1] in seq
        
        if has_a and has_b:
            # Sequence has BOTH motifs - mask ALL occurrences of BOTH at once
            loss = test_motifs_combined_cnn(seq, model, tokenizer, motifs, device)
            if loss is not None:
                results['both'].append(loss)
        elif has_a:
            # Only motif A - mask all occurrences of A
            loss = test_motifs_combined_cnn(seq, model, tokenizer, [motifs[0]], device)
            if loss is not None:
                results['motif_a'].append(loss)
        elif has_b:
            # Only motif B - mask all occurrences of B
            loss = test_motifs_combined_cnn(seq, model, tokenizer, [motifs[1]], device)
            if loss is not None:
                results['motif_b'].append(loss)
        
        # Test background (1 random chunk per sequence, avg motif length)
        motif_regions = set()
        for motif in motifs:
            for m in re.finditer(motif, seq):
                motif_regions.update(range(m.start(), m.end()))
        
        available = [pos for pos in range(len(seq)-avg_motif_len) if not any(p in motif_regions for p in range(pos, pos+avg_motif_len))]
        if available:
            start = np.random.choice(available)
            bg_chunk = seq[start:start+avg_motif_len*2]
            bg_loss = test_motifs_combined_cnn(seq, model, tokenizer, [bg_chunk], device)
            if bg_loss is not None:
                results['background'].append(bg_loss)
    
    return results

# Run analysis
print("Analyzing CNN model performance...")
cnn_results = analyze_dataset_cnn(test_data, model, tokenizer, device)

# Print counts
print(f"\nCNN Results - Motif A only: {len(cnn_results['motif_a'])}, Motif B only: {len(cnn_results['motif_b'])}, Both: {len(cnn_results['both'])}, Background: {len(cnn_results['background'])}")

# Plot results
fig, ax = plt.subplots(figsize=(10, 6))
categories = ['Motif A\nOnly', 'Motif B\nOnly', 'Both\nMotifs', 'Background']

means = [np.mean(cnn_results[k]) if cnn_results[k] else 0 for k in ['motif_a', 'motif_b', 'both', 'background']]
stds = [np.std(cnn_results[k]) if cnn_results[k] else 0 for k in ['motif_a', 'motif_b', 'both', 'background']]

bars = ax.bar(categories, means, yerr=stds, capsize=5, alpha=0.8, 
               color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
ax.set_title('CNN Model - Cross Entropy Loss by Motif Category', fontweight='bold', fontsize=14)
ax.set_ylabel('Cross Entropy Loss', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar, m in zip(bars, means):
    if m > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
                f'{m:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('cnn_motif_cross_entropy.png', dpi=300, bbox_inches='tight')
plt.show()

# Print detailed summary
print(f"\nCNN Model Performance Summary:")
print(f"  Motif A only:  {np.mean(cnn_results['motif_a']):.3f} (n={len(cnn_results['motif_a'])})")
print(f"  Motif B only:  {np.mean(cnn_results['motif_b']):.3f} (n={len(cnn_results['motif_b'])})")
print(f"  Both motifs:   {np.mean(cnn_results['both']):.3f} (n={len(cnn_results['both'])})")
print(f"  Background:    {np.mean(cnn_results['background']):.3f} (n={len(cnn_results['background'])})")

In [ ]:
# Background-normalized comparison for CNN (better baseline than theoretical random)
fig, ax = plt.subplots(figsize=(12, 6))
categories = ['Motif A\nOnly', 'Motif B\nOnly', 'Both\nMotifs', 'Background']
x = np.arange(len(categories))
width = 0.6

# Normalize by background loss (empirical baseline for CNN)
cnn_bg_mean = np.mean(cnn_results['background'])
cnn_norm = {k: [l/cnn_bg_mean for l in v] for k, v in cnn_results.items()}

cnn_means = [np.mean(cnn_norm[k]) if cnn_norm[k] else 0 for k in ['motif_a', 'motif_b', 'both', 'background']]
cnn_stds = [np.std(cnn_norm[k]) if cnn_norm[k] else 0 for k in ['motif_a', 'motif_b', 'both', 'background']]

bars = ax.bar(x, cnn_means, width, yerr=cnn_stds, capsize=5, alpha=0.8, color='#E74C3C')

ax.set_ylabel('Motif Loss / Background Loss', fontweight='bold', fontsize=12)
ax.set_title('CNN Background-Normalized Cross-Entropy (Motif Difficulty)', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='Background Level')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    if height > 0:
        ax.text(bar.get_x() + bar.get_width()/2., height, f'{height:.3f}',
               ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('cnn_motif_cross_entropy_background_norm.png', dpi=300, bbox_inches='tight')
plt.show()

# Statistical comparison for CNN
print(f"\n{'='*60}")
print("CNN BACKGROUND-NORMALIZED CROSS-ENTROPY")
print(f"{'='*60}")
print(f"CNN background: {cnn_bg_mean:.4f}")
print(f"{'-'*60}")

print(f"\nCNN (relative to its background):")
for cat, label in [('motif_a', 'Motif A'), ('motif_b', 'Motif B'), ('both', 'Both')]:
    mean_val = np.mean(cnn_norm[cat]) if cnn_norm[cat] else 0
    interpretation = "EASIER" if mean_val < 1.0 else "HARDER"
    print(f"  {label:10s}: {mean_val:.3f}x background ({interpretation} to predict)")

print(f"\n{'='*60}")
print("INTERPRETATION:")
print("  < 1.0 = Motif is EASIER to predict than random background")
print("  > 1.0 = Motif is HARDER to predict than random background") 
print("  = 1.0 = Motif has same difficulty as background")
print(f"{'='*60}")

In [ ]:
# Test biological distance shift with CNN model
def calc_ce_both_motifs_cnn(seq, model, tokenizer, motif_a, motif_b, device):
    """Calculate cross-entropy for both motifs using CNN and average"""
    losses = []
    
    # Process Motif A
    match_a = re.search(motif_a, seq)
    if match_a:
        start = match_a.start()
        tokens = [tokenizer._convert_token_to_id(c) for c in seq if tokenizer._convert_token_to_id(c) is not None]
        input_ids = [tokenizer._convert_token_to_id('[CLS]')] + tokens + [tokenizer._convert_token_to_id('[SEP]')]
        mask_id = tokenizer._convert_token_to_id('[MASK]')
        
        for i, c in enumerate(motif_a):
            pos = start + 1 + i
            masked = input_ids.copy()
            masked[pos] = mask_id
            with torch.no_grad():
                logits = model(torch.tensor([masked], device=device), return_dict=True).logits[0]
            loss = F.cross_entropy(logits[pos].unsqueeze(0), torch.tensor([tokenizer._convert_token_to_id(c)], device=device))
            losses.append(loss.item())
    
    # Process Motif B
    match_b = re.search(motif_b, seq)
    if match_b:
        start = match_b.start()
        tokens = [tokenizer._convert_token_to_id(c) for c in seq if tokenizer._convert_token_to_id(c) is not None]
        input_ids = [tokenizer._convert_token_to_id('[CLS]')] + tokens + [tokenizer._convert_token_to_id('[SEP]')]
        mask_id = tokenizer._convert_token_to_id('[MASK]')
        
        for i, c in enumerate(motif_b):
            pos = start + 1 + i
            masked = input_ids.copy()
            masked[pos] = mask_id
            with torch.no_grad():
                logits = model(torch.tensor([masked], device=device), return_dict=True).logits[0]
            loss = F.cross_entropy(logits[pos].unsqueeze(0), torch.tensor([tokenizer._convert_token_to_id(c)], device=device))
            losses.append(loss.item())
    
    return np.mean(losses) if losses else None

# Collect sequences with both motifs for CNN testing
cnn_seqs = [(row['sequences'], re.search(motifs[0], row['sequences']).span(), re.search(motifs[1], row['sequences']).span()) 
            for _, row in test_data.iterrows() if motifs[0] in row['sequences'] and motifs[1] in row['sequences']][:50]  # Smaller sample for speed

print(f"Testing {len(cnn_seqs)} CNN sequences")

# Store results for each k
k_values = [1, 2, 3, 4, 5]
cnn_shift_results = {k: {'deltas': []} for k in k_values}

for k in k_values:
    print(f"Processing k={k}...")
    
    # CNN sequences
    for seq, pos_a, pos_b in cnn_seqs:
        gap_start, gap_end = pos_a[1], pos_b[0]
        if k > (gap_end - gap_start): continue
        
        orig = calc_ce_both_motifs_cnn(seq, model, tokenizer, motifs[0], motifs[1], device)
        if orig is None: continue
        
        gap_mid = (gap_start + gap_end) // 2
        mod_seq = seq[:gap_mid] + seq[gap_mid+k:] + ''.join(np.random.choice(['A','C','G','T'], k))
        mod = calc_ce_both_motifs_cnn(mod_seq, model, tokenizer, motifs[0], motifs[1], device)
        if mod is not None:
            cnn_shift_results[k]['deltas'].append(mod - orig)

# Print results
print(f"\n{'k':<4} {'CNN Δ':<15} {'n_samples':<10}")
print("-" * 35)
for k in k_values:
    cnn_mean = np.mean(cnn_shift_results[k]['deltas']) if cnn_shift_results[k]['deltas'] else 0
    n_samples = len(cnn_shift_results[k]['deltas'])
    print(f"{k:<4} {cnn_mean:+.4f} (±{np.std(cnn_shift_results[k]['deltas']):.4f})  {n_samples:<10}")

# Plot CNN biological distance shift results
fig, ax = plt.subplots(figsize=(10, 6))
cnn_means = [np.mean(cnn_shift_results[k]['deltas']) if cnn_shift_results[k]['deltas'] else 0 for k in k_values]
cnn_stds = [np.std(cnn_shift_results[k]['deltas']) if cnn_shift_results[k]['deltas'] else 0 for k in k_values]

ax.errorbar(k_values, cnn_means, yerr=cnn_stds, marker='s', capsize=5, label='CNN', linewidth=2, color='#E74C3C')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='No change')
ax.set_xlabel('Number of deletions (k)', fontweight='bold', fontsize=12)
ax.set_ylabel('Δ Cross-Entropy', fontweight='bold', fontsize=12)
ax.set_title('CNN: Impact of k Deletions on Both Motifs Prediction', fontweight='bold', fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('cnn_biological_distance_shift.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Minimal demo: CNN Cross-entropy with motif modifications
# Edit the sequences below to experiment with CNN!

# Test sequence
cnn_test_seq = "CGTAGGTCTCGGTCTGACGTAACGCTCCAA"

k = 2  # Number of deletions to add

# ANSI color codes
BLUE = '\033[94m'    # Motif A
GREEN = '\033[92m'   # Motif B  
RED = '\033[91m'     # Deleted region
YELLOW = '\033[93m'  # Added region
RESET = '\033[0m'

def colorize_seq(seq, pos_a, pos_b, delete_start=None, delete_end=None, add_start=None, add_end=None):
    """Add color to sequence highlighting motifs and modifications"""
    result = ""
    for i, char in enumerate(seq):
        if delete_start and delete_start <= i < delete_end:
            result += f"{RED}{char}{RESET}"
        elif add_start and add_start <= i < add_end:
            result += f"{YELLOW}{char}{RESET}"
        elif pos_a[0] <= i < pos_a[1]:
            result += f"{BLUE}{char}{RESET}"
        elif pos_b[0] <= i < pos_b[1]:
            result += f"{GREEN}{char}{RESET}"
        else:
            result += char
    return result

print("="*60)
print("CNN MODEL CROSS-ENTROPY DEMONSTRATION")
print("="*60)

# Find motif positions in CNN test sequence
pos_a = re.search(motifs[0], cnn_test_seq).span()
pos_b = re.search(motifs[1], cnn_test_seq).span() 
gap_start, gap_end = pos_a[1], pos_b[0]

# Original CNN cross-entropy
orig_cnn = calc_ce_both_motifs_cnn(cnn_test_seq, model, tokenizer, motifs[0], motifs[1], device)
gap_mid = (gap_start + gap_end) // 2
print(f"Original: {colorize_seq(cnn_test_seq, pos_a, pos_b)}")
print(f"          {BLUE}Motif A{RESET} | {GREEN}Motif B{RESET}")
print(f"CNN Cross-Entropy: {orig_cnn:.4f}\n")

# Delete k random tokens between motifs, add k random at end
random_bases = ''.join(np.random.choice(['A','C','G','T'], k))
mod_cnn_seq = cnn_test_seq[:gap_mid] + cnn_test_seq[gap_mid+k:] + random_bases

# Find new motif positions after modification
pos_a_mod = re.search(motifs[0], mod_cnn_seq).span()
pos_b_mod = (pos_b[0] - k, pos_b[1] - k)  # Shifted by k
add_start = len(mod_cnn_seq) - k

print(f"Modified: {colorize_seq(cnn_test_seq, pos_a, pos_b, gap_mid, gap_mid+k)}")
print(f"          {RED}← deleted {k} bases here{RESET}")
print(f"Modified: {colorize_seq(mod_cnn_seq, pos_a_mod, pos_b_mod, add_start=add_start, add_end=len(mod_cnn_seq))}")
print(f"          {YELLOW}← added {k} random bases at end{RESET}")

mod_cnn = calc_ce_both_motifs_cnn(mod_cnn_seq, model, tokenizer, motifs[0], motifs[1], device)
print(f"CNN Cross-Entropy: {mod_cnn:.4f}")
print(f"Δ Cross-Entropy: {mod_cnn - orig_cnn:+.4f}")

print("\n" + "="*60)
print("CNN MODEL SUMMARY:")
print(f"• Original sequence CE: {orig_cnn:.4f}")
print(f"• Modified sequence CE: {mod_cnn:.4f}")
print(f"• Change in prediction difficulty: {mod_cnn - orig_cnn:+.4f}")
if mod_cnn > orig_cnn:
    print("• CNN finds motifs HARDER to predict after modification")
else:
    print("• CNN finds motifs EASIER to predict after modification")
print("="*60)

In [ ]:
# CNN Model Analysis Summary
print(f"\n{'='*80}")
print("CNN CROSS-ENTROPY ANALYSIS COMPLETE")
print(f"{'='*80}")

print(f"\nModel Architecture:")
print(f"• CNN with {config.num_conv_layers} convolutional layers")
print(f"• Kernel size: {config.kernel_size}")
print(f"• Hidden size: {config.hidden_size}")
print(f"• Total parameters: {sum(p.numel() for p in model.parameters()):,}")

print(f"\nKey Findings:")
print(f"• Background loss: {cnn_bg_mean:.4f}")
print(f"• Motif A difficulty: {np.mean(cnn_norm['motif_a']):.3f}x background")
print(f"• Motif B difficulty: {np.mean(cnn_norm['motif_b']):.3f}x background")  
print(f"• Both motifs difficulty: {np.mean(cnn_norm['both']):.3f}x background")

print(f"\nBiological Distance Effects:")
for k in [1, 2, 3, 4, 5]:
    if cnn_shift_results[k]['deltas']:
        mean_effect = np.mean(cnn_shift_results[k]['deltas'])
        print(f"• {k} deletions: {mean_effect:+.4f} change in cross-entropy")

print(f"\n{'='*80}")
print("Use this CNN model for:")
print("• Fast inference on DNA sequences")
print("• Local pattern recognition")  
print("• Parameter-efficient masked language modeling")
print("• Motif prediction tasks")
print(f"{'='*80}")